In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42) # for reproducibility

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.EMNIST(root='./data', split='letters',
                            train=True, download=True, transform=transform)

test_data = datasets.EMNIST(root='./data', split='letters',
                           train=False, download=True, transform=transform)

#Stage 1: The Single Neuron (Perceptron) #784input=28x28

# define weight vector w and bias b

w = torch.randn(784, requires_grad=True)
b = torch.randn(1, requires_grad=True)

#flatten image

image, label = train_data[0]  #image shape will be 1,28,28 so to match it with w =784 so that x matches and we can do element wise mul=x *w
x = image.view(-1)             #-1=1D vector

#compute z

z = torch.dot(w, x) + b

#apply sigmoid activation

output = 1 / (1 + torch.exp(-z))

#binary classification =is this letter an "A" or not?

binary_target = 1 if label == 1 else 0


100%|██████████| 562M/562M [00:06<00:00, 87.7MB/s]


In [ ]:
#Stage 2 : Gradient Descent
#Define binary cross-entropy loss manually

#Used for binary classification tasks. It calculates the binary cross entropy loss between the target and the output.In code written as nn.BCELoss()
#ans-loss = -[y * log(y^) + (1-y) * log(1-y^)]

#Compute the loss for one sample

loss = -(y * torch.log(output) + (1 - y) * torch.log(1 - output))

#Call loss.backward() to compute gradients

loss.backward()

#Update weights

lr = 0.01
with torch.no_grad():
        w -= lr * w.grad     #w.grad=weight gradient=dl/dw
        b -= lr * b.grad

#zero the gradients
w.grad.zero_()
b.grad.zero_()

# Print loss
if step % 20 == 0:
        print(f"Step {step}, Loss: {loss.item()}")

#Run for at least 200 steps and print the loss every 20 steps

# Training loop
for step in range(200):

    # Forward pass
    y = torch.dot(w, x) + b
    output = 1 / (1 + torch.exp(-z))   # sigmoid

    # Binary Cross Entropy Loss (manual)
    loss = -(y * torch.log(output) + (1 - y) * torch.log(1 - output))

    # Backward pass
    loss.backward()

    # Update weights (Gradient Descent)
    lr = 0.01
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    # Zero gradients
    w.grad.zero_()
    b.grad.zero_()

    # Print loss
    if step % 20 == 0:
        print(f"Step {step}, Loss: {loss.item()}")


In [ ]:
#Stage 3 : Feedforward Neural Network
#Replace your single neuron with a proper feedforward network using nn.Module that classifies all 26 letters.
#Build a network: Input (784) → Hidden layer (256 units, ReLU) → Hidden layer (128 units,ReLU) → Output (26 units)
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt


torch.manual_seed(42) # for reproducibility

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.EMNIST(root='./data', split='letters',
                            train=True, download=True, transform=transform)

test_data = datasets.EMNIST(root='./data', split='letters',
                           train=False, download=True, transform=transform)
#Replace your single neuron with a proper feedforward network using nn.Module that classifies all 26 letters.
#Build a network: Input (784) → Hidden layer (256 units, ReLU) → Hidden layer (128 units,ReLU) → Output (26 units)

from torch.utils.data import DataLoader
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

class FFNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 26)

    def forward(self, x):
        # print("Input shape:", x.shape)

        # Flatten input
        x = x.view(-1, 784)
        # print("After flattening:", x.shape)

        # First layer
        h1 = self.fc1(x)
        # print("After fc1:", h1.shape)

        h1 = torch.relu(h1)

        # Second layer
        h2 = self.fc2(h1)
        # print("After fc2:", h2.shape)

        h2 = torch.relu(h2)

        # Output layer
        y = self.fc3(h2)
        # print("After fc3 (output):", y.shape)

        return y


#Use nn.CrossEntropyLoss and torch.optim.SGD

model = FFNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

#Train for 5 epochs using a DataLoader with batch size 64
# Print training loss after each epoch
# Report final accuracy on the test set

from torch.utils.data import DataLoader
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

train_losses = []  # Initialize train_losses list
test_losses = []


for epoch in range(5):
    total_loss = 0

    for images, labels in train_loader: # DataLoader (batch_size=64)
       # ── Fix label range: 1-26 → 0-25
        # EMNIST gives 1 to 26, so subtract 1
         labels = labels - 1
         optimizer.zero_grad()              # Clear gradients
         outputs = model(images)            # Forward pass
         loss = criterion(outputs, labels)  # Compute loss
         loss.backward()                   # Backpropagation
         optimizer.step()                  # Update weights

         total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    print(f"Epoch {epoch+1}, Loss: {avg_loss}")

# Compute test loss
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for images, labels in test_loader:
            labels = labels - 1
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

    avg_test_loss = test_loss / len(test_loader)
    test_losses.append(avg_test_loss)

    print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Test Loss = {avg_test_loss:.4f}")


# Report final accuracy on the test set
correct = 0
total = 0
with torch.no_grad(): # Disable gradient calculation for inference
    for images, labels in test_loader:
        labels = labels - 1 # Adjust labels for EMNIST (1-26 to 0-25)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

ffnn_accuracy = 100 * correct / total
print(f"Accuracy on the test set: {ffnn_accuracy:.2f}%")


In [ ]:
with torch.no_grad():
    sample_images, labels = next(iter(test_loader))
    y_pred_ffnn = model(sample_images)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ---------------- Left Plot ----------------
axes[0].imshow(sample_images[0].squeeze(), cmap="gray")
axes[0].set_title("FFNN Sample Prediction")

# ---------------- Right Plot ----------------
axes[1].plot(train_losses, label="Train Loss")
axes[1].plot(test_losses, label="Test Loss")
axes[1].set_title("FFNN Loss vs Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
#Stage 4 : Convolutional Neural Network (CNN)
#Upgrade to a CNN to better capture the spatial structure of the letter images.
#Build a CNN with: Conv layer 1: 1 input channel, 16 filters, 3×3 kernel, ReLU MaxPooling: 2×2Conv layer 2: 16 input channels, 32 filters, 3×3 kernel, ReLUMaxPooling: 2×2Flatten → Fully connected (128 units, ReLU) → Output (26 units)
# Train for 5 epochs and compare test accuracy against your Stage 3 network
#Print a small table showing: Model | Epochs | Test Accuracy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Conv layers
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3)   # 28x28 → 26x26
        self.pool = nn.MaxPool2d(2, 2)                 # → 13x13

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3)  # → 11x11
                                                        # → pool → 5x5

        # Fully connected layers
        self.fc1 = nn.Linear(32 * 5 * 5, 128)
        self.fc2 = nn.Linear(128, 26)

    def forward(self, x):
        # Conv Block 1
        x = self.pool(F.relu(self.conv1(x)))

        # Conv Block 2
        x = self.pool(F.relu(self.conv2(x)))

        # Flatten
        x = x.view(x.size(0), -1)

        # Fully connected
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

model_cnn = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model_cnn.parameters(), lr=0.01)

train_losses_cnn = []
test_losses_cnn = []

for epoch in range(5):
    total_loss = 0

    model_cnn.train()
    for images, labels in train_loader:
        labels = labels - 1
        optimizer.zero_grad()
        outputs = model_cnn(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses_cnn.append(avg_train_loss)

    # Compute test loss
    model_cnn.eval()
    test_loss = 0
    with torch.no_grad():
        for images, labels in test_loader:
            labels = labels - 1
            outputs = model_cnn(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()

    avg_test_loss = test_loss / len(test_loader)
    test_losses_cnn.append(avg_test_loss)

    print(f"CNN Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Test Loss = {avg_test_loss:.4f}")

#ACCURACY

model_cnn.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        labels = labels - 1

        outputs = model_cnn(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

cnn_accuracy = 100 * correct / total

print(f"CNN Test Accuracy: {cnn_accuracy:.2f}%")

#Print a small table showing: Model | Epochs | Test Accuracy

print("\nModel Comparison:")
print("-------------------------------")
print("Model  | Epochs | Accuracy")
print("-------------------------------")
print(f"FFNN         | 5      | {ffnn_accuracy:.2f}%")
print(f"CNN          | 5      | {cnn_accuracy:.2f}%")
print("-------------------------------")

In [ ]:
with torch.no_grad():
    sample_images, labels = next(iter(test_loader))
    y_pred_cnn = model_cnn(sample_images)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ---------------- Left Plot ----------------
axes[0].imshow(sample_images[0].squeeze(), cmap="gray")
axes[0].set_title("CNN Sample Prediction")

# ---------------- Right Plot ----------------
axes[1].plot(train_losses_cnn, label="Train Loss")
axes[1].plot(test_losses_cnn, label="Test Loss")
axes[1].set_title("CNN Loss vs Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()